# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:

import re
import math
import numpy as np
import pandas as pd
import nltk
from collections import Counter
from nltk.corpus import gutenberg
from IPython.display import display

# Descargar corpus Gutenberg
nltk.download('gutenberg', quiet=True)

# Seleccionar los mismos 10 documentos de la práctica anterior
file_ids = gutenberg.fileids()[:10]

corpus_docs = []

# Recuperar y procesar documentos
def preprocess(text):
    return re.findall(r'(?u)\b\w+\b', text.lower())

for file_id in file_ids:
    raw_text = gutenberg.raw(file_id)
    title = file_id.replace('.txt', '').replace('-', ' ').title()
    tokens = preprocess(raw_text)

    corpus_docs.append({
        'doc_id': file_id,
        'title': title,
        'text': raw_text,
        'tokens': tokens,
        'length': len(tokens)
    })

# Crear tabla del corpus
df_corpus = pd.DataFrame(corpus_docs)

# Calcular DF de cada término
N = len(corpus_docs)
df_counter = Counter()

for doc in corpus_docs:
    df_counter.update(set(doc['tokens']))

# Calcular IDF para BM25
df_terms = pd.DataFrame([
    {
        'Término': term,
        'DF': df_value,
        'IDF': math.log(1 + (N - df_value + 0.5) / (df_value + 0.5))
    }
    for term, df_value in df_counter.items()
])

# Mostrar tabla de documentos
display(df_corpus[['doc_id', 'title', 'text', 'tokens']])

# Mostrar DF e IDF
display(df_terms.sort_values('DF', ascending=False).reset_index(drop=True))

print(f"Número de documentos: {N}")
print(f"Tamaño del vocabulario: {len(df_terms)}")
print(f"Longitud promedio: {np.mean([d['length'] for d in corpus_docs]):.2f}")

,doc_id,title,text,tokens
0,austen-emma.txt,Austen Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAP...,"[emma, by, jane, austen, 1816, volume, i, chap..."
1,austen-persuasion.txt,Austen Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter ...,"[persuasion, by, jane, austen, 1818, chapter, ..."
2,austen-sense.txt,Austen Sense,[Sense and Sensibility by Jane Austen 1811]\n\...,"[sense, and, sensibility, by, jane, austen, 18..."
3,bible-kjv.txt,Bible Kjv,[The King James Bible]\n\nThe Old Testament of...,"[the, king, james, bible, the, old, testament,..."
4,blake-poems.txt,Blake Poems,[Poems by William Blake 1789]\n\n \nSONGS OF I...,"[poems, by, william, blake, 1789, songs, of, i..."
5,bryant-stories.txt,Bryant Stories,[Stories to Tell to Children by Sara Cone Brya...,"[stories, to, tell, to, children, by, sara, co..."
6,burgess-busterbrown.txt,Burgess Busterbrown,[The Adventures of Buster Bear by Thornton W. ...,"[the, adventures, of, buster, bear, by, thornt..."
7,carroll-alice.txt,Carroll Alice,[Alice's Adventures in Wonderland by Lewis Car...,"[alice, s, adventures, in, wonderland, by, lew..."
8,chesterton-ball.txt,Chesterton Ball,[The Ball and The Cross by G.K. Chesterton 190...,"[the, ball, and, the, cross, by, g, k, chester..."
9,chesterton-brown.txt,Chesterton Brown,[The Wisdom of Father Brown by G. K. Chesterto...,"[the, wisdom, of, father, brown, by, g, k, che..."


,Término,DF,IDF
0,terror,10,0.04652
1,saying,10,0.04652
2,just,10,0.04652
3,against,10,0.04652
4,others,10,0.04652
...,...,...,...
25397,serenaded,1,1.99243
25398,epoch,1,1.99243
25399,slum,1,1.99243
25400,paddled,1,1.99243


Número de documentos: 10
Tamaño del vocabulario: 25402
Longitud promedio: 147447.60


Se recuperaron los mismos 10 documentos utilizados en la práctica anterior y se aplicó el mismo preprocesamiento para obtener sus tokens normalizados. Posteriormente, se calculó DF y el IDF de cada término, obteniendo las estadísticas necesarias para implementar BM25.

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Definir las ocho consultas
queries = [
    {
        'query': 'the and of',
        'description': 'Buscar documentos utilizando términos muy frecuentes.'
    },
    {
        'query': 'love marriage',
        'description': 'Encontrar historias sobre amor y matrimonio.'
    },
    {
        'query': 'alice rabbit wonderland',
        'description': 'Encontrar relatos relacionados con Alicia y el País de las Maravillas.'
    },
    {
        'query': 'god faith heaven',
        'description': 'Buscar contenido religioso.'
    },
    {
        'query': 'sea ship sailors adventure',
        'description': 'Encontrar historias de navegación y aventuras marítimas.'
    },
    {
        'query': 'father brown detective mystery',
        'description': 'Buscar relatos de detectives y misterios.'
    },
    {
        'query': 'sensibility persuasion affection romantic relationships',
        'description': 'Encontrar novelas sobre relaciones sentimentales.'
    },
    {
        'query': 'william blake poems imagination eternity',
        'description': 'Buscar poesía relacionada con la imaginación y la eternidad.'
    }
]

# 2. Mostrar las consultas y el DF de sus términos
query_info = []

for q in queries:
    for term in preprocess(q['query']):
        query_info.append({
            'Consulta': q['query'],
            'Término': term,
            'DF': df_counter.get(term, 0)
        })

df_query_terms = pd.DataFrame(query_info)

display(pd.DataFrame(queries))
display(df_query_terms)

# 3. Preparar TF-IDF conservando la matriz dispersa
docs_as_str = [' '.join(d['tokens']) for d in corpus_docs]

vectorizer = TfidfVectorizer(
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm='l2',
    analyzer=lambda x: x.split()
)

tfidf_matrix = vectorizer.fit_transform(docs_as_str)

# 4. Parámetros de BM25
k1 = 1.2
b = 0.75

avgdl = np.mean([d['length'] for d in corpus_docs])

# 5. Función para calcular BM25
def calculate_bm25(query, doc, k1=1.2, b=0.75):
    score = 0.0
    term_counts = Counter(doc['tokens'])
    query_terms = set(preprocess(query))

    for term in query_terms:
        tf = term_counts.get(term, 0)
        df = df_counter.get(term, 0)

        if tf > 0 and df > 0:
            idf = math.log(
                1 + (N - df + 0.5) / (df + 0.5)
            )

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (
                1 - b + b * doc['length'] / avgdl
            )

            score += idf * (numerator / denominator)

    return score

# 6. Calcular rankings para cada consulta
results = {}

for q_idx, q in enumerate(queries):

    query_text = q['query']
    query_processed = ' '.join(preprocess(query_text))

    # Ranking TF-IDF + coseno
    query_vector = vectorizer.transform([query_processed])
    cosine_scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    tfidf_ranking = []

    for i, score in enumerate(cosine_scores):
        if score > 0:
            tfidf_ranking.append({
                'doc_id': str(corpus_docs[i]['doc_id']),
                'title': corpus_docs[i]['title'],
                'score': score
            })

    tfidf_ranking.sort(
        key=lambda x: (-x['score'], x['doc_id'])
    )

    # Ranking BM25
    bm25_ranking = []

    for doc in corpus_docs:
        score = calculate_bm25(query_text, doc, k1, b)

        if score > 0:
            bm25_ranking.append({
                'doc_id': str(doc['doc_id']),
                'title': doc['title'],
                'score': score
            })

    bm25_ranking.sort(
        key=lambda x: (-x['score'], x['doc_id'])
    )

    # Conservar hasta diez resultados positivos
    results[query_text] = {
        'description': q['description'],
        'tfidf': tfidf_ranking[:10],
        'bm25': bm25_ranking[:10]
    }

    print(f"\n{'='*70}")
    print(f"CONSULTA {q_idx + 1}: {query_text}")
    print(f"Objetivo: {q['description']}")

    print("\nRanking TF-IDF + Coseno:")
    display(pd.DataFrame(tfidf_ranking[:10]))

    print("\nRanking BM25:")
    display(pd.DataFrame(bm25_ranking[:10]))

,query,description
0,the and of,Buscar documentos utilizando términos muy frec...
1,love marriage,Encontrar historias sobre amor y matrimonio.
2,alice rabbit wonderland,Encontrar relatos relacionados con Alicia y el...
3,god faith heaven,Buscar contenido religioso.
4,sea ship sailors adventure,Encontrar historias de navegación y aventuras ...
5,father brown detective mystery,Buscar relatos de detectives y misterios.
6,sensibility persuasion affection romantic rela...,Encontrar novelas sobre relaciones sentimentales.
7,william blake poems imagination eternity,Buscar poesía relacionada con la imaginación y...


,Consulta,Término,DF
0,the and of,the,10
1,the and of,and,10
2,the and of,of,10
3,love marriage,love,10
4,love marriage,marriage,6
5,alice rabbit wonderland,alice,1
6,alice rabbit wonderland,rabbit,5
7,alice rabbit wonderland,wonderland,1
8,god faith heaven,god,8
9,god faith heaven,faith,7



CONSULTA 1: the and of
Objetivo: Buscar documentos utilizando términos muy frecuentes.

Ranking TF-IDF + Coseno:


,doc_id,title,score
0,bible-kjv.txt,Bible Kjv,0.810712
1,blake-poems.txt,Blake Poems,0.747803
2,bryant-stories.txt,Bryant Stories,0.710732
3,chesterton-brown.txt,Chesterton Brown,0.705496
4,chesterton-ball.txt,Chesterton Ball,0.699897
5,austen-persuasion.txt,Austen Persuasion,0.629354
6,carroll-alice.txt,Carroll Alice,0.568909
7,austen-sense.txt,Austen Sense,0.566658
8,austen-emma.txt,Austen Emma,0.553972
9,burgess-busterbrown.txt,Burgess Busterbrown,0.476588



Ranking BM25:


,doc_id,title,score
0,bible-kjv.txt,Bible Kjv,0.306996
1,chesterton-ball.txt,Chesterton Ball,0.306952
2,austen-emma.txt,Austen Emma,0.306949
3,austen-sense.txt,Austen Sense,0.306946
4,austen-persuasion.txt,Austen Persuasion,0.306945
5,chesterton-brown.txt,Chesterton Brown,0.306945
6,bryant-stories.txt,Bryant Stories,0.306913
7,carroll-alice.txt,Carroll Alice,0.306855
8,burgess-busterbrown.txt,Burgess Busterbrown,0.306771
9,blake-poems.txt,Blake Poems,0.306612


... (24 outputs left out)


CONSULTA 8: william blake poems imagination eternity
Objetivo: Buscar poesía relacionada con la imaginación y la eternidad.

Ranking TF-IDF + Coseno:


,doc_id,title,score
0,blake-poems.txt,Blake Poems,0.014384
1,austen-emma.txt,Austen Emma,0.000993
2,carroll-alice.txt,Carroll Alice,0.000917
3,austen-sense.txt,Austen Sense,0.000672
4,austen-persuasion.txt,Austen Persuasion,0.000546
5,chesterton-brown.txt,Chesterton Brown,0.000324
6,chesterton-ball.txt,Chesterton Ball,0.000153
7,bible-kjv.txt,Bible Kjv,0.000080



Ranking BM25:


,doc_id,title,score
0,blake-poems.txt,Blake Poems,7.185466
1,austen-persuasion.txt,Austen Persuasion,3.696824
2,chesterton-brown.txt,Chesterton Brown,3.034105
3,austen-emma.txt,Austen Emma,2.183346
4,austen-sense.txt,Austen Sense,2.029702
5,chesterton-ball.txt,Chesterton Ball,1.861700
6,bible-kjv.txt,Bible Kjv,1.480900
7,carroll-alice.txt,Carroll Alice,0.796124


Se diseñaron ocho consultas en inglés de distinta longitud y temática, seleccionadas según su frecuencia documental (df) para abarcar términos tanto comunes como específicos. Con ello se evaluaron y ordenaron de forma descendente los top 10 resultados positivos de TF-IDF (similitud coseno) y BM25 (k1 = 1.2, b = 0.75), estableciendo una base comparativa entre ambos modelos.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [7]:
for query_text, data in results.items():
    tfidf = data['tfidf']
    bm25 = data['bm25']

    tfidf_ids = [d['doc_id'] for d in tfidf]
    bm25_ids = [d['doc_id'] for d in bm25]

    shared = set(tfidf_ids) & set(bm25_ids)

    print("\n" + "=" * 100)
    print(f"CONSULTA: {query_text}")
    print(f"Documentos compartidos en el Top 10: {len(shared)}")
    print(f"Resultados TF-IDF: {len(tfidf)}")
    print(f"Resultados BM25: {len(bm25)}")

    print("\nPrimer resultado TF-IDF:",
          tfidf[0]['title'] if tfidf else "Sin resultados")

    print("Primer resultado BM25:",
          bm25[0]['title'] if bm25 else "Sin resultados")

    tfidf_positions = {
        d['doc_id']: i + 1 for i, d in enumerate(tfidf)
    }
    bm25_positions = {
        d['doc_id']: i + 1 for i, d in enumerate(bm25)
    }

    comparison = []

    all_ids = sorted(set(tfidf_ids) | set(bm25_ids))

    for doc_id in all_ids:
        title = next(
            d['title'] for d in tfidf + bm25
            if d['doc_id'] == doc_id
        )

        comparison.append({
            'ID y título': f"{doc_id} - {title}",
            'Posición TF-IDF': tfidf_positions.get(doc_id, '-'),
            'Posición BM25': bm25_positions.get(doc_id, '-')
        })

    display(pd.DataFrame(comparison))


CONSULTA: the and of
Documentos compartidos en el Top 10: 10
Resultados TF-IDF: 10
Resultados BM25: 10

Primer resultado TF-IDF: Bible Kjv
Primer resultado BM25: Bible Kjv


,ID y título,Posición TF-IDF,Posición BM25
0,austen-emma.txt - Austen Emma,9,3
1,austen-persuasion.txt - Austen Persuasion,6,5
2,austen-sense.txt - Austen Sense,8,4
3,bible-kjv.txt - Bible Kjv,1,1
4,blake-poems.txt - Blake Poems,2,10
5,bryant-stories.txt - Bryant Stories,3,7
6,burgess-busterbrown.txt - Burgess Busterbrown,10,9
7,carroll-alice.txt - Carroll Alice,7,8
8,chesterton-ball.txt - Chesterton Ball,5,2
9,chesterton-brown.txt - Chesterton Brown,4,6



CONSULTA: love marriage
Documentos compartidos en el Top 10: 10
Resultados TF-IDF: 10
Resultados BM25: 10

Primer resultado TF-IDF: Blake Poems
Primer resultado BM25: Austen Sense


,ID y título,Posición TF-IDF,Posición BM25
0,austen-emma.txt - Austen Emma,3,3
1,austen-persuasion.txt - Austen Persuasion,4,2
2,austen-sense.txt - Austen Sense,2,1
3,bible-kjv.txt - Bible Kjv,5,4
4,blake-poems.txt - Blake Poems,1,5
5,bryant-stories.txt - Bryant Stories,7,6
6,burgess-busterbrown.txt - Burgess Busterbrown,10,10
7,carroll-alice.txt - Carroll Alice,9,9
8,chesterton-ball.txt - Chesterton Ball,6,7
9,chesterton-brown.txt - Chesterton Brown,8,8



CONSULTA: alice rabbit wonderland
Documentos compartidos en el Top 10: 5
Resultados TF-IDF: 5
Resultados BM25: 5

Primer resultado TF-IDF: Carroll Alice
Primer resultado BM25: Carroll Alice


,ID y título,Posición TF-IDF,Posición BM25
0,bryant-stories.txt - Bryant Stories,3,3
1,burgess-busterbrown.txt - Burgess Busterbrown,2,2
2,carroll-alice.txt - Carroll Alice,1,1
3,chesterton-ball.txt - Chesterton Ball,4,4
4,chesterton-brown.txt - Chesterton Brown,5,5


... (6 outputs left out)


CONSULTA: sensibility persuasion affection romantic relationships
Documentos compartidos en el Top 10: 6
Resultados TF-IDF: 6
Resultados BM25: 6

Primer resultado TF-IDF: Austen Sense
Primer resultado BM25: Austen Sense


,ID y título,Posición TF-IDF,Posición BM25
0,austen-emma.txt - Austen Emma,2,3
1,austen-persuasion.txt - Austen Persuasion,3,2
2,austen-sense.txt - Austen Sense,1,1
3,bible-kjv.txt - Bible Kjv,6,6
4,chesterton-ball.txt - Chesterton Ball,5,4
5,chesterton-brown.txt - Chesterton Brown,4,5



CONSULTA: william blake poems imagination eternity
Documentos compartidos en el Top 10: 8
Resultados TF-IDF: 8
Resultados BM25: 8

Primer resultado TF-IDF: Blake Poems
Primer resultado BM25: Blake Poems


,ID y título,Posición TF-IDF,Posición BM25
0,austen-emma.txt - Austen Emma,2,4
1,austen-persuasion.txt - Austen Persuasion,5,2
2,austen-sense.txt - Austen Sense,4,5
3,bible-kjv.txt - Bible Kjv,8,7
4,blake-poems.txt - Blake Poems,1,1
5,carroll-alice.txt - Carroll Alice,3,8
6,chesterton-ball.txt - Chesterton Ball,7,6
7,chesterton-brown.txt - Chesterton Brown,6,3


Al contrastar los top 10 de ambos métodos, se observó que coinciden en varios documentos relevantes pero difieren en su ordenamiento. El análisis detallado de un par de documentos demostró cómo la saturación de frecuencia, la rareza del término (df) y la normalización por longitud alteran los puntajes individuales y redefinen las posiciones en BM25.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [11]:
query_text = 'william blake poems imagination eternity'

selected_doc_ids = [
    'austen-persuasion.txt',
    'carroll-alice.txt'
]

configurations = [
    ('Base', 1.2, 0.75),
    ('Sin ajuste por longitud', 1.2, 0.0),
    ('Mayor efecto de las repeticiones', 2.0, 0.75)
]

query_terms = preprocess(query_text)

for config_name, current_k1, current_b in configurations:

    print("\n" + "=" * 100)
    print(f"CONFIGURACIÓN: {config_name}")
    print(f"k1 = {current_k1} | b = {current_b}")

    ranking = []

    for doc in corpus_docs:
        score = calculate_bm25(
            query_text,
            doc,
            current_k1,
            current_b
        )

        if score > 0:
            ranking.append({
                'doc_id': str(doc['doc_id']),
                'title': doc['title'],
                'score': score
            })

    ranking.sort(key=lambda x: (-x['score'], x['doc_id']))

    for doc_id in selected_doc_ids:
        position = next(
            (i + 1 for i, d in enumerate(ranking)
             if d['doc_id'] == doc_id),
            '-'
        )

        doc = next(d for d in corpus_docs if d['doc_id'] == doc_id)
        print(f"\nDocumento: {doc['title']}")
        print(f"Longitud |d|: {doc['length']}")
        print(f"Posición: {position}")

        term_counts = Counter(doc['tokens'])
        contributions = []

        for term in query_terms:
            tf = term_counts.get(term, 0)
            df = df_counter.get(term, 0)

            idf = math.log(
                1 + (N - df + 0.5) / (df + 0.5)
            )

            if tf > 0 and df > 0:
                numerator = tf * (current_k1 + 1)
                denominator = tf + current_k1 * (
                    1 - current_b +
                    current_b * doc['length'] / avgdl
                )
                contribution = idf * numerator / denominator
            else:
                contribution = 0.0

            contributions.append({
                'Término': term,
                'f(t,d)': tf,
                'df(t)': df,
                'Aporte BM25': round(contribution, 6)
            })

        display(pd.DataFrame(contributions))

        print(
            "Score BM25:",
            round(calculate_bm25(
                query_text, doc, current_k1, current_b
            ), 6)
        )


CONFIGURACIÓN: Base
k1 = 1.2 | b = 0.75

Documento: Austen Persuasion
Longitud |d|: 84167
Posición: 2


,Término,"f(t,d)",df(t),Aporte BM25
0,william,4,7,0.700148
1,blake,0,1,0.000000
2,poems,1,2,1.797128
3,imagination,3,5,1.199549
4,eternity,0,3,0.000000


Score BM25: 3.696824

Documento: Carroll Alice
Longitud |d|: 27336
Posición: 8


,Término,"f(t,d)",df(t),Aporte BM25
0,william,8,7,0.796124
1,blake,0,1,0.000000
2,poems,0,2,0.000000
3,imagination,0,5,0.000000
4,eternity,0,3,0.000000


Score BM25: 0.796124

CONFIGURACIÓN: Sin ajuste por longitud
k1 = 1.2 | b = 0.0

Documento: Austen Persuasion
Longitud |d|: 84167
Posición: 2


,Término,"f(t,d)",df(t),Aporte BM25
0,william,4,7,0.648141
1,blake,0,1,0.000000
2,poems,1,2,1.481605
3,imagination,3,5,1.089231
4,eternity,0,3,0.000000


... (3 outputs left out)

,Término,"f(t,d)",df(t),Aporte BM25
0,william,4,7,0.858047
1,blake,0,1,0.000000
2,poems,1,2,1.886401
3,imagination,3,5,1.432043
4,eternity,0,3,0.000000


Score BM25: 4.176492

Documento: Carroll Alice
Longitud |d|: 27336
Posición: 8


,Término,"f(t,d)",df(t),Aporte BM25
0,william,8,7,1.047131
1,blake,0,1,0.000000
2,poems,0,2,0.000000
3,imagination,0,5,0.000000
4,eternity,0,3,0.000000


Score BM25: 1.047131


Manteniendo fija una consulta, se probaron tres configuraciones en BM25: base (k1 = 1.2, b = 0.75), sin normalización por longitud (b = 0) y con mayor peso a las repeticiones (k1 = 2.0). Esto evidenció cómo variar ambos hiperparámetros modifica el aporte por término, impactando directamente las puntuaciones finales y el orden del ranking.